In [1]:
import pandas as pd

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.head()

# getting rid of " " in TotalCharges
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)# make it numeric, if cannot put NaN(this is what errors="coerce" does)

#we saw that when Total Charges " " tenure=0 meaning new customers so we can
#make TotalCharges=0 for them
df["TotalCharges"] = df["TotalCharges"].fillna(0) #replace NaNs with 0s

df["Churn"] = df["Churn"].map({
    "No":0,
    "Yes":1
})

# drop meaningless feature and target to create X
X = df.drop(columns=["Churn", "customerID"])
# choosing Churn as the target
y = df["Churn"]

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [3]:
X.isna().sum().sum() #no missing value after preprocessing

np.int64(0)

In [4]:
categorical_features = [cname for cname in X.columns if
                    X[cname].dtype == "object"]

# Select numerical columns
numerical_features = [cname for cname in X.columns if 
                X[cname].dtype in ['int64', 'float64']]

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ],
    remainder="passthrough" #don't touch numerical features
)

## Baseline Decision Tree Pipeline

In [6]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline

dt_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(random_state=0))
])

dt_model.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [7]:
y_pred = dt_model.predict(X_test)
y_prob = dt_model.predict_proba(X_test)[:, 1]

## Model(DT) Evaluation

In [10]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

y_pred = dt_model.predict(X_test)
y_prob = dt_model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)
print("ROC-AUC:", roc_auc)

Accuracy: 0.7189496096522356
Precision: 0.46464646464646464
Recall: 0.5
F1-score: 0.4816753926701571
ROC-AUC: 0.6522783276949421


In [11]:
cm = confusion_matrix(y_test, y_pred)

print(cm)

[[829 212]
 [184 184]]


### Baseline Decision Tree

#### The default Decision Tree performed substantially worse than Logistic Regression across all evaluated metrics. Its ROC-AUC of 0.65 indicates limited ability to distinguish between customers who churn and those who do not.

In [12]:
# let's check if reason for that is overfitting
print("Train accuracy:", dt_model.score(X_train, y_train))
print("Test accuracy:", dt_model.score(X_test, y_test))

Train accuracy: 0.997515086971956
Test accuracy: 0.7189496096522356


#### well yes, an overfit

## Hyperparameter Tuning

In [14]:
param_grid = {
    "classifier__max_depth": [3, 5, 7, 10, 15, 20],
    "classifier__min_samples_split": [2, 5, 10, 20],
    "classifier__min_samples_leaf": [1, 2, 5, 10],
    "classifier__criterion": ["gini", "entropy"]
}

### GridSearchCV

In [16]:
from sklearn.model_selection import GridSearchCV

grid_search = GridSearchCV(
    estimator=dt_model,
    param_grid=param_grid,
    cv=5, # crossvalidation, now we are not tuning for one train-test split
    scoring="roc_auc",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

,estimator,Pipeline(step...om_state=0))])
,param_grid,"{'classifier__criterion': ['gini', 'entropy'], 'classifier__max_depth': [3, 5, ...], 'classifier__min_samples_leaf': [1, 2, ...], 'classifier__min_samples_split': [2, 5, ...]}"
,scoring,'roc_auc'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('cat', ...)]"


In [17]:
print("Best parameters:", grid_search.best_params_)
print("Best CV ROC-AUC:", grid_search.best_score_) # CV roc score, not test

Best parameters: {'classifier__criterion': 'entropy', 'classifier__max_depth': 5, 'classifier__min_samples_leaf': 5, 'classifier__min_samples_split': 20}
Best CV ROC-AUC: 0.8329650862421609


### Tuned Model

In [20]:
best_dt = grid_search.best_estimator_

#### Evaluation

In [21]:
y_pred_tuned = best_dt.predict(X_test)
y_prob_tuned = best_dt.predict_proba(X_test)[:, 1]

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("Accuracy:", accuracy_score(y_test, y_pred_tuned))
print("Precision:", precision_score(y_test, y_pred_tuned))
print("Recall:", recall_score(y_test, y_pred_tuned))
print("F1-score:", f1_score(y_test, y_pred_tuned))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_tuned))

Accuracy: 0.78708303761533
Precision: 0.60828025477707
Recall: 0.5190217391304348
F1-score: 0.5601173020527859
ROC-AUC: 0.8124360460259783


In [22]:
print("Train accuracy:", best_dt.score(X_train, y_train))
print("Test accuracy:", best_dt.score(X_test, y_test))

Train accuracy: 0.8068867589634363
Test accuracy: 0.78708303761533


### Hyperparameter tuning substantially improved the Decision Tree's performance and reduced overfitting.

## Model Comparision

In [23]:
comparison = pd.DataFrame({
    "Decision Tree": [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        f1_score(y_test, y_pred),
        roc_auc_score(y_test, y_prob)
    ],
    "Tuned Decision Tree": [
        accuracy_score(y_test, y_pred_tuned),
        precision_score(y_test, y_pred_tuned),
        recall_score(y_test, y_pred_tuned),
        f1_score(y_test, y_pred_tuned),
        roc_auc_score(y_test, y_prob_tuned)
    ]
}, index=[
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "ROC-AUC"
])

comparison.round(4)

,Decision Tree,Tuned Decision Tree
Accuracy,0.7189,0.7871
Precision,0.4646,0.6083
Recall,0.5000,0.5190
F1-score,0.4817,0.5601
ROC-AUC,0.6523,0.8124


## Conclusion

### The baseline Decision Tree showed substantial overfitting, achieving nearly perfect training accuracy but considerably lower performance on the test set. Hyperparameter tuning substantially improved its generalization performance, increasing the test ROC-AUC from 0.65 to 0.81.

### However, the tuned Decision Tree still performed slightly below the Logistic Regression baseline, which achieved a ROC-AUC of approximately 0.83. This suggests that the additional complexity of a single decision tree does not necessarily provide better predictive performance for this dataset.